# Tilling Dataset

* **Products used:** 
[ s2_l2a](https://explorer.digitalearth.africa/s2_l2a)

## Background
 

## Description

***

## Getting started
To run this analysis, run all the cells in the notebook, starting with the "Load packages" cell. 

### Load packages


In [1]:
%matplotlib inline

import datacube
import geopandas as gpd
from odc.geo.geom import Geometry

import os
import gc
import math
import logging
import psutil
import numpy as np
import dask.array as da
from pathlib import Path
from typing import Callable, Optional, Tuple, Union
import rasterio
from rasterio.transform import from_bounds
from rasterio.windows import Window

from deafrica_tools.datahandling import load_ard
from deafrica_tools.plotting import rgb
from deafrica_tools.areaofinterest import define_area

In [ ]:
"""
Tiled Datacube Processor
========================
Processes large dask-loaded datacube images in memory-safe tiles.
Each tile is processed independently and saved as a separate image file,
preventing kernel crashes on large datasets.
"""



#  Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# Memory helpers
def _available_memory_mb() -> float:
    """Return available system RAM in MB."""
    return psutil.virtual_memory().available / (1024 ** 2)


def _estimate_tile_memory_mb(
    tile_rows: int,
    tile_cols: int,
    n_bands: int,
    dtype: np.dtype,
    safety_factor: float = 3.0,
) -> float:
    """
    Estimate MB needed to hold one tile (raw + processing headroom).

    ``safety_factor`` accounts for intermediate arrays created during
    processing (default 3× raw size is a conservative estimate).
    """
    bytes_per_pixel = np.dtype(dtype).itemsize
    raw_mb = (tile_rows * tile_cols * n_bands * bytes_per_pixel) / (1024 ** 2)
    return raw_mb * safety_factor


def _compute_safe_tile_size(
    total_rows: int,
    total_cols: int,
    n_bands: int,
    dtype: np.dtype,
    max_memory_fraction: float = 0.25,
    min_tile_size: int = 128,
    max_tile_size: int = 4096,
) -> int:
    """
    Auto-calculate a square tile side-length that fits within
    ``max_memory_fraction`` of available RAM.

    Returns an integer tile size (pixels per side).
    """
    avail_mb = _available_memory_mb()
    budget_mb = avail_mb * max_memory_fraction

    bytes_per_pixel = np.dtype(dtype).itemsize
    # pixels = budget_bytes / (bands * bytes * safety_factor)
    safety = 3.0
    max_pixels = (budget_mb * 1024 ** 2) / (n_bands * bytes_per_pixel * safety)
    tile_side = int(math.sqrt(max_pixels))

    tile_side = max(min_tile_size, min(tile_side, max_tile_size))
    tile_side = min(tile_side, total_rows, total_cols)  # can't exceed image dims

    log.info(
        "Memory budget: %.0f MB available → %.0f MB reserved per tile "
        "→ tile_size=%d px",
        avail_mb,
        budget_mb,
        tile_side,
    )
    return tile_side


def process_tiles(
    datacube: da.Array,
    process_fn: Callable[[np.ndarray], np.ndarray],
    output_dir: Union[str, Path],
    *,
    tile_size: Optional[int] = None,
    overlap: int = 0,
    max_memory_fraction: float = 0.25,
    output_format: str = "GTiff",
    output_dtype: Optional[str] = None,
    nodata: Optional[float] = None,
    crs: Optional[str] = None,
    transform=None,
    file_prefix: str = "tile",
) -> list[Path]:
    """
    Process a dask datacube by dividing it into spatial tiles.

    Each tile is:
      1. Pulled from dask into RAM as a numpy array.
      2. Passed through ``process_fn``.
      3. Written to disk as a GeoTIFF (or other GDAL format).
      4. Evicted from memory before the next tile begins.

    Parameters
    ----------
    datacube : dask.array.Array
    process_fn : callable
        Signature: ``np.ndarray → np.ndarray``
        Receives a tile of shape (bands, tile_rows, tile_cols) and must
        return an array of shape (out_bands, tile_rows, tile_cols) or
        (tile_rows, tile_cols).
    output_dir : str | Path
        Directory where tile images are saved (created if absent).
    tile_size : int, optional
        Side length in pixels. Auto-computed from available RAM if None.
    overlap : int
        Pixel overlap between adjacent tiles (for algorithms that need
        neighbourhood context). Overlap pixels are *included* in saved tiles.
    max_memory_fraction : float
        Fraction of available RAM to use per tile (default 0.25 = 25 %).
    output_format : str
        GDAL driver name (default "GTiff").
    output_dtype : str, optional
        Output numpy dtype string (e.g. "float32"). Inferred if None.
    nodata : float, optional
        NoData value written to output metadata.
    crs : str, optional
        Coordinate reference system (e.g. "EPSG:4326").
    transform : affine.Affine, optional
        Affine transform for the full image; tile transforms are derived
        automatically.
    file_prefix : str
        Prefix for output filenames (default "tile").

    Returns
    -------
    list[Path]
        Paths of all written tile files (row-major order).
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # ── Normalise to (bands, rows, cols) ─────────────────────────────────────
    if datacube.ndim == 2:
        datacube = datacube[np.newaxis, ...]   # add fake band axis
        squeezed = True
    elif datacube.ndim == 3:
        squeezed = False
    else:
        raise ValueError(
            f"datacube must be 2-D (rows, cols) or 3-D (bands, rows, cols); "
            f"got shape {datacube.shape}"
        )

    n_bands, total_rows, total_cols = datacube.shape
    src_dtype = datacube.dtype

    # ── Tile size ─────────────────────────────────────────────────────────────
    if tile_size is None:
        tile_size = _compute_safe_tile_size(
            total_rows, total_cols, n_bands, src_dtype, max_memory_fraction
        )
    else:
        log.info("Using user-supplied tile_size=%d px", tile_size)

    step = tile_size - overlap  # stride between tile origins
    if step <= 0:
        raise ValueError("overlap must be smaller than tile_size")

    row_starts = list(range(0, total_rows, step))
    col_starts = list(range(0, total_cols, step))
    total_tiles = len(row_starts) * len(col_starts)
    log.info(
        "Image: %d bands × %d rows × %d cols → %d tiles (%d×%d grid)",
        n_bands, total_rows, total_cols,
        total_tiles, len(row_starts), len(col_starts),
    )

    written: list[Path] = []
    tile_idx = 0

    for ri, row0 in enumerate(row_starts):
        for ci, col0 in enumerate(col_starts):
            tile_idx += 1
            row1 = min(row0 + tile_size, total_rows)
            col1 = min(col0 + tile_size, total_cols)

            log.info(
                "Tile %d/%d  rows[%d:%d] cols[%d:%d]  "
                "(%.0f MB free)",
                tile_idx, total_tiles,
                row0, row1, col0, col1,
                _available_memory_mb(),
            )

            # ── 1. Pull tile from dask ────────────────────────────────────
            tile_dask = datacube[:, row0:row1, col0:col1]
            tile_np: np.ndarray = tile_dask.compute()   # single compute call

            # ── 2. Process ───────────────────────────────────────────────
            if squeezed:
                # pass (rows, cols) if original was 2-D
                result = process_fn(tile_np[0])
                if result.ndim == 2:
                    result = result[np.newaxis, ...]
            else:
                result = process_fn(tile_np)
                if result.ndim == 2:
                    result = result[np.newaxis, ...]

            if result.ndim != 3:
                raise ValueError(
                    f"process_fn must return 2-D or 3-D array; got shape {result.shape}"
                )

            out_bands, out_rows, out_cols = result.shape
            out_dtype = np.dtype(output_dtype) if output_dtype else result.dtype

            # ── 3. Derive tile geo-transform ─────────────────────────────
            if transform is not None:
                tile_transform = transform * transform.translation(col0, row0)
            else:
                tile_transform = None

            # ── 4. Write tile ────────────────────────────────────────────
            ext = ".tif" if output_format == "GTiff" else f".{output_format.lower()}"
            filename = f"{file_prefix}_r{ri:04d}_c{ci:04d}{ext}"
            out_path = output_dir / filename

            profile = {
                "driver": output_format,
                "dtype": str(out_dtype),
                "width": out_cols,
                "height": out_rows,
                "count": out_bands,
            }
            if nodata is not None:
                profile["nodata"] = nodata
            if crs is not None:
                profile["crs"] = crs
            if tile_transform is not None:
                profile["transform"] = tile_transform
            if output_format == "GTiff":
                profile.update({"compress": "lzw", "tiled": True})

            with rasterio.open(out_path, "w", **profile) as dst:
                dst.write(result.astype(out_dtype))

            written.append(out_path)
            log.info("  → saved %s", out_path.name)

            # ── 5. Explicit memory release ───────────────────────────────
            del tile_np, tile_dask, result
            gc.collect()

    log.info("Done. %d tiles written to %s", len(written), output_dir)
    return written

### Connect to the datacube

In [2]:
dc = datacube.Datacube(app='tilling')

## Load Sentinel-2 data from the datacube

Here we are loading in a timeseries of Sentinel-2 satellite images through the datacube API.
This will provide us with some data to work with.

The following cell sets the parameters, which define the area of interest and the length of time to conduct the analysis over.
The parameters are

* `lat`: The central latitude to analyse (e.g. `6.502`).
* `lon`: The central longitude to analyse (e.g. `-1.409`).
* `buffer`: The number of square degrees to load around the central latitude and longitude.
For reasonable loading times, set this as `0.1` or lower.


#### Select location
To define the area of interest, there are two methods available:

1. By specifying the latitude, longitude, and buffer. This method requires you to input the central latitude, central longitude, and the buffer value in square degrees around the center point you want to analyze. For example, `lat = 10.338`, `lon = -1.055`, and `buffer = 0.1` will select an area with a radius of 0.1 square degrees around the point with coordinates (10.338, -1.055).

2. By uploading a polygon as a `GeoJSON or Esri Shapefile`. If you choose this option, you will need to upload the geojson or ESRI shapefile into the Sandbox using Upload Files button <img align="top" src="../Supplementary_data/upload_files_icon.png"> in the top left corner of the Jupyter Notebook interface. ESRI shapefiles must be uploaded with all the related files `(.cpg, .dbf, .shp, .shx)`. Once uploaded, you can use the shapefile or geojson to define the area of interest. Remember to update the code to call the file you have uploaded.

To use one of these methods, you can uncomment the relevant line of code and comment out the other one. To comment out a line, add the `"#"` symbol before the code you want to comment out. By default, the first option which defines the location using latitude, longitude, and buffer is being used.

In [3]:
# Set the area of interest

# Method 1: Specify the latitude, longitude, and buffer
aoi = define_area(lat=13.94, lon=-16.54, buffer=0.05)

# Method 2: Use a polygon as a GeoJSON or Esri Shapefile. 
# aoi = define_area(vector_path='aoi.shp')

#Create a geopolygon and geodataframe of the area of interest
geopolygon = Geometry(aoi["features"][0]["geometry"], crs="epsg:4326")
geopolygon_gdf = gpd.GeoDataFrame(geometry=[geopolygon], crs=geopolygon.crs)

# Get the latitude and longitude range of the geopolygon
lat_range = (geopolygon_gdf.total_bounds[1], geopolygon_gdf.total_bounds[3])
lon_range = (geopolygon_gdf.total_bounds[0], geopolygon_gdf.total_bounds[2])

# Create a reusable query
query = {
    'x': lon_range,
    'y': lat_range,
    'time': ('2020-11-01', '2020-12-15'),
    'resolution': (-20, 20),
    'measurements': ['red', 'green', 'blue'],
    'output_crs':'EPSG:6933'
}

# Load available data from Landsat 8 and filter to retain only times
# with at least 50% good data
ds = load_ard(dc=dc, 
              products=['s2_l2a'], 
              **query)

# Print output data
print(ds)


Using pixel quality parameters for Sentinel 2
Finding datasets
    s2_l2a
Applying pixel quality/cloud mask
Loading 9 time steps
<xarray.Dataset> Size: 32MB
Dimensions:      (time: 9, y: 620, x: 483)
Coordinates:
  * time         (time) datetime64[ns] 72B 2020-11-03T11:47:42 ... 2020-12-13...
  * y            (y) float64 5kB 1.768e+06 1.768e+06 ... 1.755e+06 1.755e+06
  * x            (x) float64 4kB -1.601e+06 -1.601e+06 ... -1.591e+06 -1.591e+06
    spatial_ref  int32 4B 6933
Data variables:
    red          (time, y, x) float32 11MB nan nan nan nan ... 110.0 131.0 140.0
    green        (time, y, x) float32 11MB nan nan nan nan ... 247.0 248.0 290.0
    blue         (time, y, x) float32 11MB nan nan nan nan ... 67.0 87.0 89.0
Attributes:
    crs:           EPSG:6933
    grid_mapping:  spatial_ref


In [ ]:
# Define any per-tile processing function 
def my_processing(tile: np.ndarray) -> np.ndarray:
    """
    Example: compute NDVI from bands 3 (NIR) and 2 (Red).
    Replace with your real algorithm.

    tile shape: (bands, rows, cols)
    """
    nir  = tile[3].astype(np.float32)
    red  = tile[2].astype(np.float32)
    ndvi = (nir - red) / (nir + red + 1e-8)
    return ndvi  # shape: (rows, cols) — function handles 2-D returns

#Run tiled processor 
tile_paths = process_tiles(
    datacube=ds,
    process_fn=my_processing,
    output_dir="./output_tiles",
    tile_size=None,             # auto-detect from available RAM
    overlap=32,                 # 32-px border overlap if needed
    max_memory_fraction=0.25,   # use at most 25 % of free RAM per tile
    output_dtype="float32",
    nodata=-9999.0,
    crs="EPSG:4326",
    file_prefix="ndvi",
)

print(f"\nProcessed {len(tile_paths)} tiles:")
for p in tile_paths[:5]:
    print(" ", p)
if len(tile_paths) > 5:
    print(f"  ... and {len(tile_paths) - 5} more")

---

## Additional information

**License:** The code in this notebook is licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0). 
Digital Earth Africa data is licensed under the [Creative Commons by Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) license.

**Contact:** If you need assistance, please post a question on the [Open Data Cube Slack channel](http://slack.opendatacube.org/) or on the [GIS Stack Exchange](https://gis.stackexchange.com/questions/ask?tags=open-data-cube) using the `open-data-cube` tag (you can view previously asked questions [here](https://gis.stackexchange.com/questions/tagged/open-data-cube)).
If you would like to report an issue with this notebook, you can file one on [Github](https://github.com/digitalearthafrica/deafrica-sandbox-notebooks).

**Compatible datacube version:** 

In [10]:
print(datacube.__version__)

1.8.20


**Last Tested:**

In [11]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d')

'2025-01-15'